# RAG — Elasticsearch Pipeline

This notebook demonstrates a RAG pipeline using **Elasticsearch** as the search backend.

Pipeline steps:
1. **Load** FAQ documents via HTTP (`FaqHttpLoader`)
2. **Index** documents into Elasticsearch (`ElasticsearchIndex.index_docs`)
3. **Query** the pipeline with a natural-language question (`RAGBase`)
4. **Answer** is generated via OpenRouter (`OpenRouterClient`)

## Start Elasticsearch

Run Elasticsearch 8.x in Docker (security disabled for local dev):

```bash
docker run --rm -d \
  --name elasticsearch \
  -p 9200:9200 \
  -e "discovery.type=single-node" \
  -e "xpack.security.enabled=false" \
  docker.elastic.co/elasticsearch/elasticsearch:8.17.6
```

Wait ~20 seconds, then verify it's up:

```bash
curl http://localhost:9200
```

> **Note:** Requires `OPENROUTER_API_KEY` in your `.env` file.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.insert(0, '..')

from src import FaqHttpLoader, ElasticsearchIndex, RAGBase, OpenRouterClient

In [3]:
loader = FaqHttpLoader()
docs = loader.load()
print(f"Loaded {len(docs)} documents")

Loaded 1208 documents


In [4]:
index = ElasticsearchIndex(host="http://localhost:9200", index_name="faq")
index.index_docs(docs)
print(f"Indexed {len(docs)} documents into Elasticsearch")

Indexed 1208 documents into Elasticsearch


In [5]:
assistant = RAGBase(
    index=index,
    llm=OpenRouterClient(),
    model="openrouter/owl-alpha",
    course_filter="llm-zoomcamp"
)

In [6]:
answer = assistant.rag("How do I join the course?")
print(answer)

To join the course, you can simply start learning and submit your homework while the submission form is open. There is no need to wait for a confirmation email or register beforehand—registration is only to gauge interest before the course starts. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.
